In [4]:
# 04_evaluation.ipynb - 模型評估：測試集分析與穩定性檢驗
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

print("✓ 導入完成")

✓ 導入完成


In [5]:
# 載入數據
processed_path = "../data/processed/elliptic_processed.pt"
data = torch.load(processed_path)
x = data['x']
edge_index = data['edge_index']
y = data['y']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = x.to(device)
edge_index = edge_index.to(device)
y = y.to(device)

print(f"✓ 數據已載入，使用設備: {device}")
print(f"  特徵維度: {x.shape}, 邊數: {edge_index.shape[1]}, 節點數: {y.shape[0]}")

✓ 數據已載入，使用設備: cpu
  特徵維度: torch.Size([203769, 167]), 邊數: 234355, 節點數: 203769


/var/folders/6p/f3bb7wps1p9crqh3cv0vjjfc0000gn/T/ipykernel_38581/3332245717.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(processed_path)


In [6]:
# 拆分訓練/驗證/測試集
labeled_mask = y != -1
labeled_indices = torch.where(labeled_mask)[0].cpu().numpy()

# 第一次拆分：80% (train+val) 和 20% (test)
train_val_idx, test_idx = train_test_split(
    labeled_indices, test_size=0.2,
    stratify=y[labeled_mask].cpu().numpy(), random_state=42
)

# 第二次拆分：從 train_val 中再拆 80% train 和 20% val
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2,
    stratify=y[train_val_idx].cpu().numpy(), random_state=42
)

train_idx = torch.tensor(train_idx, device=device)
val_idx = torch.tensor(val_idx, device=device)
test_idx = torch.tensor(test_idx, device=device)

print(f"✓ 數據拆分完成")
print(f"  訓練: {len(train_idx)}, 驗證: {len(val_idx)}, 測試: {len(test_idx)}")
print(f"  比例: Train {len(train_idx)/len(labeled_indices)*100:.1f}% | Val {len(val_idx)/len(labeled_indices)*100:.1f}% | Test {len(test_idx)/len(labeled_indices)*100:.1f}%")

✓ 數據拆分完成
  訓練: 29800, 驗證: 7451, 測試: 9313
  比例: Train 64.0% | Val 16.0% | Test 20.0%


In [7]:
# 定義 GraphSAGE 模型
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden=128, num_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_feats, hidden))
        for _ in range(1, num_layers):
            self.layers.append(nn.Linear(hidden * 2, hidden))
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, x, edge_index, idx=None):
        h = x
        for i, layer in enumerate(self.layers):
            row, col = edge_index
            num_neigh = torch.bincount(row, minlength=h.size(0)).float()
            aggr = torch.zeros_like(h)
            aggr.index_add_(dim=0, index=row, source=h[col])
            aggr[row.unique()] /= (num_neigh[row.unique()] + 1e-10).unsqueeze(1)
            
            if i == 0:
                h = F.relu(layer(h))
            else:
                h_cat = torch.cat([h, aggr], dim=1)
                h = F.relu(layer(h_cat))
                                   
        logits = self.classifier(h)
        if idx is not None:
            return logits[idx]
        return logits

print("✓ GraphSAGE 模型已定義")

✓ GraphSAGE 模型已定義


In [8]:
# 定義訓練函數
def train_and_evaluate(seed, num_epochs=100, verbose=False):
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    train_val_idx_new, test_idx_new = train_test_split(
        labeled_indices, test_size=0.2,
        stratify=y[labeled_mask].cpu().numpy(), random_state=seed
    )
    
    train_idx_new, val_idx_new = train_test_split(
        train_val_idx_new, test_size=0.2,
        stratify=y[train_val_idx_new].cpu().numpy(), random_state=seed
    )
    
    train_idx_new = torch.tensor(train_idx_new, device=device)
    val_idx_new = torch.tensor(val_idx_new, device=device)
    test_idx_new = torch.tensor(test_idx_new, device=device)
    
    model_new = GraphSAGE(in_feats=x.shape[1]).to(device)
    optimizer_new = Adam(model_new.parameters(), lr=0.01)
    loss_fn_new = nn.CrossEntropyLoss()
    
    best_val_auc_new = 0.0
    patience_counter_new = 0
    patience = 5
    
    for epoch in range(num_epochs):
        model_new.train()
        optimizer_new.zero_grad()
        logits_new = model_new(x, edge_index, train_idx_new)
        loss_new = loss_fn_new(logits_new, y[train_idx_new])
        loss_new.backward()
        optimizer_new.step()
        
        if epoch % 10 == 0:
            model_new.eval()
            with torch.no_grad():
                val_logits_new = model_new(x, edge_index, val_idx_new)
                val_probs_new = F.softmax(val_logits_new, dim=1)[:, 1].cpu().numpy()
                val_labels_new = y[val_idx_new].cpu().numpy()
                val_auc_new = roc_auc_score(val_labels_new, val_probs_new)
            
            if val_auc_new > best_val_auc_new:
                best_val_auc_new = val_auc_new
                patience_counter_new = 0
                best_model_state_new = model_new.state_dict().copy()
            else:
                patience_counter_new += 1
                if patience_counter_new >= patience:
                    model_new.load_state_dict(best_model_state_new)
                    if verbose:
                        print(f"Seed {seed}: Early stopping at epoch {epoch}")
                    break
    
    model_new.eval()
    with torch.no_grad():
        logits_new = model_new(x, edge_index)
        probs_new = F.softmax(logits_new, dim=1)[:, 1].cpu().numpy()
    
    test_probs_new = probs_new[test_idx_new.cpu().numpy()]
    test_labels_new = y[test_idx_new].cpu().numpy()
    test_auc_new = roc_auc_score(test_labels_new, test_probs_new)
    test_auprc_new = average_precision_score(test_labels_new, test_probs_new)
    
    return test_auc_new, test_auprc_new, epoch + 1, test_labels_new

print("✓ 訓練函數已定義")

✓ 訓練函數已定義


In [9]:
# 執行多次實驗
print("\n" + "="*60)
print("穩定性分析：運行 10 次不同 Random Seed")
print("="*60 + "\n")

results = []
seeds = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555]

for i, seed in enumerate(tqdm(seeds, desc="實驗進度")):
    test_auc_i, test_auprc_i, epochs_i, test_labels_i = train_and_evaluate(seed, num_epochs=100, verbose=False)
    results.append({
        'Seed': seed,
        'Test AUC': test_auc_i,
        'Test AUPRC': test_auprc_i,
        'Epochs': epochs_i
    })
    
    if i == 0:
        test_labels = test_labels_i
        test_auc = test_auc_i
        test_auprc = test_auprc_i

results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("實驗結果詳情")
print("="*60)
print(results_df.to_string(index=False))


穩定性分析：運行 10 次不同 Random Seed



實驗進度:   0%|          | 0/10 [01:18<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# 統計分析
print("\n" + "="*60)
print("統計分析")
print("="*60)

auc_mean = results_df['Test AUC'].mean()
auc_std = results_df['Test AUC'].std()
auc_min = results_df['Test AUC'].min()
auc_max = results_df['Test AUC'].max()

auprc_mean = results_df['Test AUPRC'].mean()
auprc_std = results_df['Test AUPRC'].std()
auprc_min = results_df['Test AUPRC'].min()
auprc_max = results_df['Test AUPRC'].max()

print(f"\nTest AUC 統計：")
print(f"  平均值: {auc_mean:.4f}")
print(f"  標準差: {auc_std:.4f}")
print(f"  95% 信賴區間: [{auc_mean - 1.96*auc_std:.4f}, {auc_mean + 1.96*auc_std:.4f}]")
print(f"  最小值 - 最大值: {auc_min:.4f} - {auc_max:.4f}")
print(f"  變異係數 (CV): {auc_std/auc_mean*100:.2f}%")

print(f"\nTest AUPRC 統計：")
print(f"  平均值: {auprc_mean:.4f}")
print(f"  標準差: {auprc_std:.4f}")
print(f"  95% 信賴區間: [{auprc_mean - 1.96*auprc_std:.4f}, {auprc_mean + 1.96*auprc_std:.4f}]")
print(f"  最小值 - 最大值: {auprc_min:.4f} - {auprc_max:.4f}")
print(f"  變異係數 (CV): {auprc_std/auprc_mean*100:.2f}%")

print(f"\n訓練 epochs 統計：")
print(f"  平均 epochs: {results_df['Epochs'].mean():.1f}")
print(f"  最小 epochs: {results_df['Epochs'].min()}")
print(f"  最大 epochs: {results_df['Epochs'].max()}")

if auc_std < 0.01:
    stability = "✓ 極穩定（std < 0.01）"
elif auc_std < 0.02:
    stability = "✓ 穩定（std < 0.02）"
else:
    stability = "⚠️  較不穩定（std >= 0.02），建議增加訓練或調整超參數"

print(f"\n穩定性評估：{stability}")

In [ ]:
# 分析測試集類別分布
print("\n" + "="*60)
print("測試集數據分析")
print("="*60)

test_normal = np.sum(test_labels == 0)
test_anomaly = np.sum(test_labels == 1)
test_total = len(test_labels)
anomaly_ratio = test_anomaly / test_total * 100

print(f"\n類別分布：")
print(f"  正常樣本（標籤 0）: {test_normal} ({test_normal/test_total*100:.1f}%)")
print(f"  異常樣本（標籤 1）: {test_anomaly} ({anomaly_ratio:.1f}%)")
print(f"  總樣本數: {test_total}")

print(f"\n類別不平衡比：{test_normal / max(test_anomaly, 1):.2f}:1 (Normal:Anomaly)")

print(f"\n💡 指標解釋：")
print(f"  • AUC-ROC: {test_auc:.4f} (全局評估，對不平衡敏感度低)")
print(f"  • AUPRC: {test_auprc:.4f} (重點評估少數類，對不平衡敏感度高)")

if anomaly_ratio < 10:
    print(f"\n⚠️  異常樣本比例只有 {anomaly_ratio:.1f}%，建議重視 AUPRC 指標")
elif anomaly_ratio < 50:
    print(f"\n✓ 中等類別不平衡（{anomaly_ratio:.1f}%），AUPRC 和 AUC 都應參考")
else:
    print(f"\n✓ 類別相對平衡（{anomaly_ratio:.1f}%），可主要參考 AUC")

In [ ]:
# 可視化結果
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 圖1：AUC 分布箱線圖
ax1 = axes[0, 0]
bp = ax1.boxplot([results_df['Test AUC'], results_df['Test AUPRC']], 
                   labels=['AUC-ROC', 'AUPRC'],
                   patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('Distribution of Metrics Across 10 Seeds', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# 圖2：AUC 隨 Seed 的變化
ax2 = axes[0, 1]
colors = plt.cm.viridis(np.linspace(0, 1, len(results_df)))
ax2.scatter(range(len(results_df)), results_df['Test AUC'], s=100, alpha=0.7, color=colors)
ax2.axhline(y=auc_mean, color='r', linestyle='--', linewidth=2, label=f'Mean: {auc_mean:.4f}')
ax2.fill_between(range(len(results_df)), auc_mean - auc_std, auc_mean + auc_std, 
                  alpha=0.2, color='red', label=f'±1 Std: {auc_std:.4f}')
ax2.set_xlabel('Experiment Index', fontsize=11)
ax2.set_ylabel('Test AUC', fontsize=11)
ax2.set_title('Test AUC Across Different Seeds', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 圖3：AUPRC 隨 Seed 的變化
ax3 = axes[1, 0]
ax3.scatter(range(len(results_df)), results_df['Test AUPRC'], s=100, alpha=0.7, color=colors)
ax3.axhline(y=auprc_mean, color='g', linestyle='--', linewidth=2, label=f'Mean: {auprc_mean:.4f}')
ax3.fill_between(range(len(results_df)), auprc_mean - auprc_std, auprc_mean + auprc_std,
                  alpha=0.2, color='green', label=f'±1 Std: {auprc_std:.4f}')
ax3.set_xlabel('Experiment Index', fontsize=11)
ax3.set_ylabel('Test AUPRC', fontsize=11)
ax3.set_title('Test AUPRC Across Different Seeds', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 圖4：直方圖
ax4 = axes[1, 1]
ax4.hist(results_df['Test AUC'], bins=6, alpha=0.6, color='blue', label='AUC-ROC', edgecolor='black')
ax4.hist(results_df['Test AUPRC'], bins=6, alpha=0.6, color='orange', label='AUPRC', edgecolor='black')
ax4.axvline(auc_mean, color='blue', linestyle='--', linewidth=2, alpha=0.7)
ax4.axvline(auprc_mean, color='orange', linestyle='--', linewidth=2, alpha=0.7)
ax4.set_xlabel('Score', fontsize=11)
ax4.set_ylabel('Frequency', fontsize=11)
ax4.set_title('Distribution of Scores Across 10 Experiments', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("結論")
print("="*60)
print(f"""
1. 異常比例：{anomaly_ratio:.1f}%（{'高度不平衡' if anomaly_ratio < 10 else '中等不平衡' if anomaly_ratio < 50 else '相對平衡'}）
2. AUC-ROC 穩定性：μ={auc_mean:.4f}, σ={auc_std:.4f}, CV={auc_std/auc_mean*100:.2f}%
3. AUPRC 穩定性：μ={auprc_mean:.4f}, σ={auprc_std:.4f}, CV={auprc_std/auprc_mean*100:.2f}%
4. 模型穩定狀態：{stability}
5. 平均訓練 epochs：{results_df['Epochs'].mean():.1f}（說明 early stopping 有效）
""")